<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Image_Color_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image colour analysis — median RGB, CIE Lab (D65), and Munsell (Illuminant C)

Upload one ZIP archive manually. The notebook analyses every supported image within it, computes the alpha-weighted median RGB channel values, reports CIE Lab relative to D65, and obtains Munsell notation after Bradford/Von Kries adaptation from D65 to Illuminant C. Results are exported to a formatted Excel workbook.

In [5]:
#@title 1. Install and import required packages
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'PIL': 'Pillow',
    'colour': 'colour-science',
    'pandas': 'pandas',
    'openpyxl': 'openpyxl',
}
missing_packages = [
    package for module, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])

from datetime import datetime
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import re
import warnings

import colour
from colour.adaptation import chromatic_adaptation_VonKries
from colour.utilities import ColourUsageWarning
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from google.colab import files

warnings.filterwarnings('ignore', category=ColourUsageWarning)
print('Dependencies are ready.')

Dependencies are ready.


In [6]:
#@title 2. Set analysis options
REPORT_TITLE = 'Median Image Colour Analysis'
SUPPORTED_IMAGE_EXTENSIONS = {'.bmp', '.gif', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}
OBSERVER = 'CIE 1931 2 Degree Standard Observer'
CAT_TRANSFORM = 'Bradford'

# RGB and CIE Lab are referenced to D65. Munsell renotation data are defined under C.
D65_XY = colour.CCS_ILLUMINANTS[OBSERVER]['D65']
C_XY = colour.CCS_ILLUMINANTS[OBSERVER]['C']
D65_WHITE_XYZ = colour.xy_to_XYZ(D65_XY)
C_WHITE_XYZ = colour.xy_to_XYZ(C_XY)

print('RGB / CIE Lab illuminant: D65')
print('Munsell illuminant: C')
print('Chromatic adaptation: Von Kries with Bradford transform')
print('Supported formats:', ', '.join(sorted(SUPPORTED_IMAGE_EXTENSIONS)))

RGB / CIE Lab illuminant: D65
Munsell illuminant: C
Chromatic adaptation: Von Kries with Bradford transform
Supported formats: .bmp, .gif, .jpeg, .jpg, .png, .tif, .tiff, .webp


In [7]:
#@title 3. Define image-analysis, colour-conversion, and Excel-export functions
HUE_SECTORS = ['R', 'YR', 'Y', 'GY', 'G', 'BG', 'B', 'PB', 'P', 'RP']

def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f'Unsafe ZIP member path: {member.filename}')
        archive.extractall(destination)

def median_rgb(image_path):
    """Return alpha-weighted median 8-bit RGB values from the first image frame."""
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image)
        rgba = np.asarray(image.convert('RGBA'), dtype=np.uint8).reshape(-1, 4)
    visible = rgba[rgba[:, 3] > 0, :3]
    if visible.size == 0:
        raise ValueError('The image has no non-transparent pixels.')
    return tuple(np.rint(np.median(visible, axis=0)).astype(int))

def format_munsell_number(number):
    number = float(number)
    if np.isclose(number, round(number)):
        return str(int(round(number)))
    return f'{number:.2f}'.rstrip('0').rstrip('.')

def normalise_zero_hue_to_previous_sector(munsell_notation, tolerance=1e-8):
    text = str(munsell_notation).strip().upper()
    match = re.fullmatch(
        r'([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*([A-Z]+)\s+([^\s/]+)\s*/\s*([^\s]+)',
        text,
    )
    if match is None:
        return text
    hue_number_text, sector, value_text, chroma_text = match.groups()
    hue_number = float(hue_number_text)
    if np.isclose(hue_number, 0.0, atol=tolerance) and sector in HUE_SECTORS:
        prior_sector = HUE_SECTORS[(HUE_SECTORS.index(sector) - 1) % len(HUE_SECTORS)]
        return f'10{prior_sector} {value_text}/{chroma_text}'
    return f'{format_munsell_number(hue_number)}{sector} {value_text}/{chroma_text}'

def round_to_half(value):
    return np.floor(float(value) * 2.0 + 0.5) / 2.0

def format_half(value):
    value = round_to_half(value)
    return str(int(round(value))) if np.isclose(value, round(value)) else f'{value:.1f}'

def munsell_to_half_step(code):
    neutral = re.fullmatch(r'N\s*([0-9.]+)', str(code).strip().upper())
    if neutral:
        return f'N {format_half(neutral.group(1))}'
    chromatic = re.fullmatch(r'([0-9.]+)\s*([A-Z]+)\s+([0-9.]+)\s*/\s*([0-9.]+)', str(code).strip().upper())
    if chromatic is None:
        return str(code)
    hue, sector, value, chroma = chromatic.groups()
    return f'{format_half(hue)}{sector} {format_half(value)}/{format_half(chroma)}'

def rgb_to_lab_and_munsell(rgb):
    """Convert median sRGB to Lab(D65), then Munsell(C) via Bradford/Von Kries CAT."""
    rgb_normalised = np.asarray(rgb, dtype=float) / 255.0
    xyz_d65 = colour.sRGB_to_XYZ(rgb_normalised)
    lab_d65 = colour.XYZ_to_Lab(xyz_d65, illuminant=D65_XY)

    if np.max(np.abs(lab_d65[1:])) < 1.5:
        return tuple(np.round(lab_d65, 2)), f'N {lab_d65[0]:.1f}', ''

    # The Munsell conversion must receive XYZ referenced to Illuminant C.
    xyz_c = chromatic_adaptation_VonKries(
        xyz_d65, D65_WHITE_XYZ, C_WHITE_XYZ, transform=CAT_TRANSFORM
    )
    xyY_c = colour.XYZ_to_xyY(xyz_c)
    try:
        raw_code = colour.xyY_to_munsell_colour(xyY_c)
        return tuple(np.round(lab_d65, 2)), normalise_zero_hue_to_previous_sector(raw_code), ''
    except Exception:
        # Use only a minimally altered C-referenced coordinate for an approximate Munsell code.
        lab_for_search = np.array([np.clip(lab_d65[0], 10.0, 90.0), lab_d65[1], lab_d65[2]])
        lightness_clipped = not np.isclose(lab_for_search[0], lab_d65[0])
        for chroma_scale in np.linspace(1.0, 0.0, 101):
            candidate_lab = np.array([lab_for_search[0], lab_for_search[1] * chroma_scale, lab_for_search[2] * chroma_scale])
            candidate_xyz_d65 = colour.Lab_to_XYZ(candidate_lab, illuminant=D65_XY)
            candidate_xyz_c = chromatic_adaptation_VonKries(
                candidate_xyz_d65, D65_WHITE_XYZ, C_WHITE_XYZ, transform=CAT_TRANSFORM
            )
            try:
                code = colour.xyY_to_munsell_colour(colour.XYZ_to_xyY(candidate_xyz_c))
                comment = 'Approximate Munsell code: nearest valid renotation coordinate used'
                if lightness_clipped:
                    comment += '; lightness was constrained to the Munsell conversion range'
                return tuple(np.round(lab_d65, 2)), normalise_zero_hue_to_previous_sector(code), comment
            except Exception:
                continue
        return tuple(np.round(lab_d65, 2)), 'Unavailable', 'No valid Munsell renotation coordinate could be found'

def roman_to_integer(roman):
    values = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total, previous = 0, 0
    for character in reversed(roman):
        value = values[character]
        total += -value if value < previous else value
        previous = max(previous, value)
    return total

def identifier_sort_key(row):
    identifier = row['ID']
    match = re.fullmatch(r'([IVXLCDM]+)([a-z]?)', identifier, flags=re.IGNORECASE)
    if match:
        roman, suffix = match.groups()
        return (0, suffix.casefold(), roman_to_integer(roman.upper()), identifier.casefold())
    return (1, identifier.casefold(), 0, identifier.casefold())

def build_excel(rows, output_path, title):
    columns = ['ID', 'R', 'G', 'B', 'L*', 'a*', 'b*', 'Munsell code', 'Munsell code (0.5)', 'Munsell comment', 'Median RGB colour']
    dataframe = pd.DataFrame(rows, columns=columns)
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        dataframe.to_excel(writer, sheet_name='Colour analysis', index=False, startrow=2)
        worksheet = writer.book['Colour analysis']
        worksheet['A1'] = title
        worksheet['A2'] = 'Median RGB channels; CIE Lab(D65); Munsell(C) obtained by Bradford/Von Kries adaptation from D65 to C.'
        worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(columns))
        worksheet.merge_cells(start_row=2, start_column=1, end_row=2, end_column=len(columns))
        worksheet['A1'].font = Font(bold=True, size=14)
        worksheet['A2'].alignment = Alignment(wrap_text=True)
        header_row = 3
        for cell in worksheet[header_row]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        colour_column = columns.index('Median RGB colour') + 1
        for row_index, row in enumerate(rows, start=header_row + 1):
            colour_cell = worksheet.cell(row=row_index, column=colour_column)
            r, g, b = row['R'], row['G'], row['B']
            colour_cell.fill = PatternFill(fill_type='solid', fgColor=f'{r:02X}{g:02X}{b:02X}')
            colour_cell.alignment = Alignment(horizontal='center', vertical='center')
        widths = [14, 9, 9, 9, 10, 10, 10, 20, 22, 58, 22]
        for index, width in enumerate(widths, start=1):
            worksheet.column_dimensions[get_column_letter(index)].width = width
        worksheet.freeze_panes = 'A4'
        worksheet.auto_filter.ref = f'A3:{get_column_letter(len(columns))}{header_row + len(rows)}'
        for row in worksheet.iter_rows(min_row=4, max_row=header_row + len(rows), min_col=1, max_col=len(columns)):
            for cell in row:
                cell.alignment = Alignment(vertical='center', wrap_text=True)

def analyse_zip(zip_path, source_name):
    with TemporaryDirectory() as temporary_directory:
        extraction_directory = Path(temporary_directory) / 'images'
        extraction_directory.mkdir()
        safe_extract_zip(zip_path, extraction_directory)
        image_paths = sorted(
            path for path in extraction_directory.rglob('*')
            if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
        )
        if not image_paths:
            raise ValueError('No supported image files were found in the ZIP archive.')
        rows, failed_files = [], []
        for image_path in image_paths:
            try:
                rgb = median_rgb(image_path)
                lab, munsell_code, comment = rgb_to_lab_and_munsell(rgb)
                rows.append({
                    'ID': image_path.stem, 'R': rgb[0], 'G': rgb[1], 'B': rgb[2],
                    'L*': lab[0], 'a*': lab[1], 'b*': lab[2],
                    'Munsell code': munsell_code,
                    'Munsell code (0.5)': munsell_to_half_step(munsell_code),
                    'Munsell comment': comment,
                    'Median RGB colour': '',
                })
            except Exception as error:
                failed_files.append(f'{image_path.name}: {error}')
        if not rows:
            raise RuntimeError('No images could be analysed. ' + ' | '.join(failed_files))
        rows.sort(key=identifier_sort_key)
        safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', Path(source_name).stem) or 'images'
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_path = Path('/content') / f'{safe_name}_median_colour_analysis_{timestamp}.xlsx'
        build_excel(rows, output_path, f'{REPORT_TITLE}: {Path(source_name).stem}')
        preview = pd.DataFrame(rows).drop(columns=['Median RGB colour'])
        return output_path, preview, failed_files

In [8]:
#@title 4. Upload a ZIP archive manually and download the Excel report
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one ZIP file containing the images to analyse.')

zip_name = zip_names[0]
zip_path = Path('/content') / zip_name
zip_path.write_bytes(uploaded[zip_name])
output_path, preview, failed_files = analyse_zip(zip_path, zip_name)

if failed_files:
    print('Files skipped because they could not be read:')
    print('\n'.join(failed_files))
display(preview)
print(f'Excel report created: {output_path.name}')
files.download(str(output_path))

Saving QQ.zip to QQ.zip


,ID,R,G,B,L*,a*,b*,Munsell code,Munsell code (0.5),Munsell comment
0,I,193,158,233,70.65,27.25,-32.97,3.5P 7.0/10.2,3.5P 7/10,
1,II,248,151,186,73.31,40.71,-2.51,5.1RP 7.3/9.8,5RP 7.5/10,
2,III,255,213,227,89.11,16.86,-1.37,4.4RP 8.9/4.4,4.5RP 9/4.5,
3,Ia,241,219,223,89.25,8.23,0.79,7.1RP 8.9/2.5,7RP 9/2.5,
4,IIa,255,127,138,68.22,49.60,17.11,2.2R 6.7/11.6,2R 6.5/11.5,
5,IIIa,255,255,241,99.66,-2.38,6.69,1.2GY 9.0/0.8,1GY 9/1,Approximate Munsell code: nearest valid renota...


Excel report created: QQ_median_colour_analysis_20260819_112713.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes

- No online archive is retrieved: input is exclusively a ZIP file uploaded through the final cell.
- The reported RGB values are per-channel medians of non-transparent image pixels.
- sRGB-to-XYZ and CIE Lab calculations are referenced to D65. For Munsell conversion, XYZ(D65) is adapted to XYZ(C) with Von Kries chromatic adaptation using the Bradford transform, before invoking the Munsell renotation conversion.
- If direct Munsell conversion is outside the supported renotation range, the workbook leaves the code in its own cell and places the approximation explanation in the separate **Munsell comment** cell.
- The Excel colour column is filled with the corresponding median RGB colour rather than embedding a rectangle image.